# Student Performance — Pandas Practice Exercise

**Course:** FinTech Batch 09, IIM Kozhikode
**Prepared by:** Amit Chakraborty (Group 03)

This notebook works through the Student Performance dataset, following the exercise
sheet section by section. Each section starts with the stated requirement, followed
by the code used to satisfy it and the resulting output.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/students.csv")
df.head()

,StudentID,Name,Age,City,Marks,Gender,Department
0,1,Asha,19,Delhi,85,F,Physics
1,2,Rahul,20,Mumbai,92,M,Computer Science
2,3,Meera,18,Chennai,77,F,Mathematics
3,4,Vikram,21,Bangalore,68,M,Economics
4,5,Simran,19,Delhi,88,F,Mathematics


## 1. Load and inspect the data

**Requirement:** Load `students.csv`, display the first five rows, the shape and
column names, and check data types and summary statistics.

In [2]:
# Shape (rows, columns) and column names
df.shape

(20, 7)

In [3]:
df.columns

Index(['StudentID', 'Name', 'Age', 'City', 'Marks', 'Gender', 'Department'], dtype='str')

In [4]:
# Data types — note: pandas 3.0 uses a dedicated 'str' dtype for text columns
# instead of the older generic 'object' dtype
df.dtypes

StudentID     int64
Name            str
Age           int64
City            str
Marks         int64
Gender          str
Department      str
dtype: object

In [5]:
# Summary statistics for numeric columns
df.describe()

,StudentID,Age,Marks
count,20.00000,20.000000,20.000000
mean,10.50000,19.800000,80.450000
std,5.91608,1.361114,9.599753
min,1.00000,18.000000,65.000000
25%,5.75000,19.000000,72.750000
50%,10.50000,20.000000,80.500000
75%,15.25000,21.000000,88.500000
max,20.00000,22.000000,95.000000


**Q1. How many students are in the dataset?**
**Q2. What is the average mark?**
**Q3. What are the unique departments?**

In [6]:
# Q1: Student count
student_count = df.shape[0]

# Q2: Average mark
average_mark = df["Marks"].mean()

# Q3: Unique departments
unique_departments = df["Department"].unique()

print("Number of students:", student_count)
print("Average mark:", average_mark)
print("Unique departments:", list(unique_departments))

Number of students: 20
Average mark: 80.45
Unique departments: ['Physics', 'Computer Science', 'Mathematics', 'Economics', 'Psychology']


## 2. Add a new column

**Requirement:** Create a `Result` column (Pass if Marks >= 75, else Fail) and a
`Grade` column (A: 90+, B: 80-89, C: 70-79, D: 60-69, F: below 60).

In [7]:
# Result: Pass/Fail based on a 75-mark threshold
df["Result"] = np.where(df["Marks"] >= 75, "Pass", "Fail")

# Grade: banded using pd.cut. Bin edges are one less than each band's lower
# cutoff because pd.cut includes the right edge of each bin by default.
df["Grade"] = pd.cut(
    df["Marks"],
    bins=[-float("inf"), 59, 69, 79, 89, float("inf")],
    labels=["F", "D", "C", "B", "A"]
)

df[["Marks", "Result", "Grade"]].head(10)

,Marks,Result,Grade
0,85,Pass,B
1,92,Pass,A
2,77,Pass,C
3,68,Fail,D
4,88,Pass,B
5,73,Fail,C
6,95,Pass,A
7,81,Pass,B
8,69,Fail,D
9,74,Fail,C


## 3. Add a new row

**Requirement:** Add a new student (Nisha) using `pd.concat`, and ensure her
`Result` and `Grade` are also populated.

In [8]:
new_student = {
    "StudentID": 21,
    "Name": "Nisha",
    "Age": 20,
    "City": "Kolkata",
    "Marks": 86,
    "Gender": "F",
    "Department": "Psychology"
}

df = pd.concat(
    [df, pd.DataFrame([new_student])],
    ignore_index=True
)

# Result/Grade must be recomputed for the whole frame — they don't
# update automatically just because a new row was added
df["Result"] = np.where(df["Marks"] >= 75, "Pass", "Fail")
df["Grade"] = pd.cut(
    df["Marks"],
    bins=[-float("inf"), 59, 69, 79, 89, float("inf")],
    labels=["F", "D", "C", "B", "A"]
)

df.tail(3)

,StudentID,Name,Age,City,Marks,Gender,Department,Result,Grade
18,19,Kabir,22,Hyderabad,67,M,Economics,Fail,D
19,20,Tara,18,Chennai,94,F,Computer Science,Pass,A
20,21,Nisha,20,Kolkata,86,F,Psychology,Pass,B


## 4. Practise handling missing values

**Requirement:** Introduce missing values (`Marks` at row 2, `City` at row 6),
identify them, build one version that drops affected rows and another that fills
them, then explain which approach is more appropriate for this small dataset.

In [9]:
df.loc[2, "Marks"] = None
df.loc[6, "City"] = None

# Identify missing values
df.isna().sum()

StudentID     0
Name          0
Age           0
City          1
Marks         1
Gender        0
Department    0
Result        0
Grade         0
dtype: int64

**Version A — drop all rows containing missing values (on a copy):**

In [10]:
df_dropped = df.dropna()
df_dropped.shape   # 2 rows removed, one per affected row

(19, 9)

**Version B — fill missing values instead of dropping (on a copy):**

In [11]:
df_filled = df.copy()
df_filled["City"] = df_filled["City"].fillna("Unknown")
df_filled["Marks"] = df_filled["Marks"].fillna(df_filled["Marks"].median())

# Result/Grade must be recomputed again since Marks changed
df_filled["Result"] = np.where(df_filled["Marks"] >= 75, "Pass", "Fail")
df_filled["Grade"] = pd.cut(
    df_filled["Marks"],
    bins=[-float("inf"), 59, 69, 79, 89, float("inf")],
    labels=["F", "D", "C", "B", "A"]
)

df_filled.loc[[2, 6]]

,StudentID,Name,Age,City,Marks,Gender,Department,Result,Grade
2,3,Meera,18,Chennai,82.0,F,Mathematics,Pass,B
6,7,Leela,20,Unknown,95.0,F,Computer Science,Pass,A


**Which approach is more appropriate for this dataset?**

Filling is the better choice here. Dropping would discard two entire student
records (roughly 10% of the dataset) just because one field each was missing,
losing every other piece of information about those students. Filling keeps all
21 students in the analysis: `Marks` is filled with the column median (a
low-risk, standard estimate for a single missing numeric value), and `City` is
filled with the explicit label `"Unknown"` rather than left blank, so it reads as
a deliberate, visible category rather than a silent gap that could break later
grouping or filtering. This works well because it is a small fraction of missing
data (1 out of 21 per column) in columns that are not the primary variable of
interest; if a much larger share of `Marks` were missing, filling with the
median would start distorting the analysis and dropping — or explicitly flagging
the gap — would be the more honest choice.

In [12]:
# Apply the chosen approach (fill) permanently to the working dataframe
df["Marks"] = df["Marks"].fillna(df["Marks"].median())
df["City"] = df["City"].fillna("Unknown")

df["Result"] = np.where(df["Marks"] >= 75, "Pass", "Fail")
df["Grade"] = pd.cut(
    df["Marks"],
    bins=[-float("inf"), 59, 69, 79, 89, float("inf")],
    labels=["F", "D", "C", "B", "A"]
)

df.isna().sum().sum()  # 0 — dataframe is now clean

np.int64(0)

## 5. Remove duplicates

**Requirement:** Create a duplicate row, count duplicates, remove exact
duplicates, and separately check/handle duplicate student IDs.

In [13]:
df = pd.concat([df, df.iloc[[0]]], ignore_index=True)

# Count duplicates two ways
full_row_dupes = df.duplicated().sum()
id_dupes = df["StudentID"].duplicated().sum()

print("Exact duplicate rows:", full_row_dupes)
print("Duplicate StudentIDs:", id_dupes)

Exact duplicate rows: 1
Duplicate StudentIDs: 1


In [14]:
# Remove exact duplicate rows
df_no_dupes = df.drop_duplicates()

# Remove duplicate StudentIDs, keeping the first occurrence
df_no_id_dupes = df.drop_duplicates(subset="StudentID", keep="first")

print(df_no_dupes.shape, df_no_id_dupes.shape)

(21, 9) (21, 9)


In [15]:
# Apply the cleanup back to the working dataframe
df = df.drop_duplicates().reset_index(drop=True)
df.shape

(21, 9)

## 6. Filtering exercises

**Requirement:** Build 10 filtered DataFrames using Pandas boolean indexing.

**1. Students who scored above 85**

In [16]:
high_scorers = df[df["Marks"] > 85]
high_scorers

,StudentID,Name,Age,City,Marks,Gender,Department,Result,Grade
1,2,Rahul,20,Mumbai,92.0,M,Computer Science,Pass,A
4,5,Simran,19,Delhi,88.0,F,Mathematics,Pass,B
6,7,Leela,20,Unknown,95.0,F,Computer Science,Pass,A
10,11,Anita,21,Bangalore,90.0,F,Computer Science,Pass,A
14,15,Priya,18,Mumbai,91.0,F,Computer Science,Pass,A
16,17,Neha,20,Bangalore,87.0,F,Economics,Pass,B
19,20,Tara,18,Chennai,94.0,F,Computer Science,Pass,A
20,21,Nisha,20,Kolkata,86.0,F,Psychology,Pass,B


**2. Students from Delhi**

In [17]:
delhi_students = df[df["City"] == "Delhi"]
delhi_students

,StudentID,Name,Age,City,Marks,Gender,Department,Result,Grade
0,1,Asha,19,Delhi,85.0,F,Physics,Pass,B
4,5,Simran,19,Delhi,88.0,F,Mathematics,Pass,B
7,8,Arjun,18,Delhi,81.0,M,Economics,Pass,B
13,14,Sahil,19,Delhi,83.0,M,Psychology,Pass,B


**3. Female students in Computer Science**

In [18]:
female_cs = df[(df["Gender"] == "F") & (df["Department"] == "Computer Science")]
female_cs

,StudentID,Name,Age,City,Marks,Gender,Department,Result,Grade
6,7,Leela,20,Unknown,95.0,F,Computer Science,Pass,A
10,11,Anita,21,Bangalore,90.0,F,Computer Science,Pass,A
14,15,Priya,18,Mumbai,91.0,F,Computer Science,Pass,A
19,20,Tara,18,Chennai,94.0,F,Computer Science,Pass,A


**4. Students aged 20 or older who scored at least 80**

In [19]:
age_and_marks = df[(df["Age"] >= 20) & (df["Marks"] >= 80)]
age_and_marks

,StudentID,Name,Age,City,Marks,Gender,Department,Result,Grade
1,2,Rahul,20,Mumbai,92.0,M,Computer Science,Pass,A
6,7,Leela,20,Unknown,95.0,F,Computer Science,Pass,A
10,11,Anita,21,Bangalore,90.0,F,Computer Science,Pass,A
16,17,Neha,20,Bangalore,87.0,F,Economics,Pass,B
20,21,Nisha,20,Kolkata,86.0,F,Psychology,Pass,B


**5. Students from Delhi or Mumbai**

In [20]:
delhi_or_mumbai = df[df["City"].isin(["Delhi", "Mumbai"])]
delhi_or_mumbai

,StudentID,Name,Age,City,Marks,Gender,Department,Result,Grade
0,1,Asha,19,Delhi,85.0,F,Physics,Pass,B
1,2,Rahul,20,Mumbai,92.0,M,Computer Science,Pass,A
4,5,Simran,19,Delhi,88.0,F,Mathematics,Pass,B
7,8,Arjun,18,Delhi,81.0,M,Economics,Pass,B
8,9,Rekha,19,Mumbai,69.0,F,Psychology,Fail,D
13,14,Sahil,19,Delhi,83.0,M,Psychology,Pass,B
14,15,Priya,18,Mumbai,91.0,F,Computer Science,Pass,A


**6. Students whose department is Physics or Mathematics**

In [21]:
physics_or_maths = df[df["Department"].isin(["Physics", "Mathematics"])]
physics_or_maths

,StudentID,Name,Age,City,Marks,Gender,Department,Result,Grade
0,1,Asha,19,Delhi,85.0,F,Physics,Pass,B
2,3,Meera,18,Chennai,82.0,F,Mathematics,Pass,B
4,5,Simran,19,Delhi,88.0,F,Mathematics,Pass,B
5,6,Karan,22,Pune,73.0,M,Physics,Fail,C
9,10,Dev,20,Chennai,74.0,M,Mathematics,Fail,C
12,13,Divya,20,Hyderabad,78.0,F,Physics,Pass,C
15,16,Amit,21,Chennai,72.0,M,Mathematics,Fail,C
17,18,Isha,19,Pune,80.0,F,Physics,Pass,B


**7. The five students with the highest marks**

In [22]:
top_5 = df.nlargest(5, "Marks")
top_5

,StudentID,Name,Age,City,Marks,Gender,Department,Result,Grade
6,7,Leela,20,Unknown,95.0,F,Computer Science,Pass,A
19,20,Tara,18,Chennai,94.0,F,Computer Science,Pass,A
1,2,Rahul,20,Mumbai,92.0,M,Computer Science,Pass,A
14,15,Priya,18,Mumbai,91.0,F,Computer Science,Pass,A
10,11,Anita,21,Bangalore,90.0,F,Computer Science,Pass,A


**8. Students whose names begin with A**

In [23]:
names_starting_a = df[df["Name"].str.startswith("A")]
names_starting_a

,StudentID,Name,Age,City,Marks,Gender,Department,Result,Grade
0,1,Asha,19,Delhi,85.0,F,Physics,Pass,B
7,8,Arjun,18,Delhi,81.0,M,Economics,Pass,B
10,11,Anita,21,Bangalore,90.0,F,Computer Science,Pass,A
15,16,Amit,21,Chennai,72.0,M,Mathematics,Fail,C


**9. Students who passed, sorted by marks from highest to lowest**

In [24]:
passed_sorted = df[df["Result"] == "Pass"].sort_values("Marks", ascending=False)
passed_sorted[["Name", "Marks", "Result"]]

,Name,Marks,Result
6,Leela,95.0,Pass
19,Tara,94.0,Pass
1,Rahul,92.0,Pass
14,Priya,91.0,Pass
10,Anita,90.0,Pass
4,Simran,88.0,Pass
16,Neha,87.0,Pass
20,Nisha,86.0,Pass
0,Asha,85.0,Pass
13,Sahil,83.0,Pass


**10. Students scoring between 75 and 90, inclusive**

In [25]:
mid_to_high_scorers = df[df["Marks"].between(75, 90)]
mid_to_high_scorers[["Name", "Marks"]]

,Name,Marks
0,Asha,85.0
2,Meera,82.0
4,Simran,88.0
7,Arjun,81.0
10,Anita,90.0
12,Divya,78.0
13,Sahil,83.0
16,Neha,87.0
17,Isha,80.0
20,Nisha,86.0


## 7. Make inferences

**Requirement:** Answer the following using Pandas calculations rather than
visual inspection.

**1. Which department has the highest average marks?**

In [26]:
dept_avg = df.groupby("Department")["Marks"].mean().sort_values(ascending=False)
dept_avg

Department
Computer Science    92.400000
Psychology          79.333333
Mathematics         79.000000
Physics             79.000000
Economics           73.600000
Name: Marks, dtype: float64

**2. Which city has the highest average marks?**

In [27]:
city_avg = df.groupby("City")["Marks"].agg(["mean", "count"]).sort_values("mean", ascending=False)
city_avg

,mean,count
City,,
Unknown,95.000000,1
Kolkata,86.000000,1
Delhi,84.250000,4
Mumbai,84.000000,3
Bangalore,81.666667,3
Chennai,80.500000,4
Pune,72.666667,3
Hyderabad,72.500000,2


*Note:* `"Unknown"` and `"Kolkata"` each have only 1 student behind their
average, so treating either as the top "city" would be misleading. Among cities
with more than one student, **Delhi** has the highest average.

**3. What percentage of students passed?**

In [28]:
pass_percentage = df["Result"].value_counts(normalize=True)["Pass"] * 100
pass_percentage

np.float64(66.66666666666666)

**4. Do female or male students have a higher average mark?**

In [29]:
gender_avg = df.groupby("Gender")["Marks"].mean()
gender_avg

Gender
F    85.416667
M    75.000000
Name: Marks, dtype: float64

**5. Which age group performs best on average?**

In [30]:
age_avg = df.groupby("Age")["Marks"].mean().sort_values(ascending=False)
age_avg

Age
18    87.000000
20    85.333333
19    81.000000
21    76.666667
22    68.333333
Name: Marks, dtype: float64

**6. Which department has the most students?**

In [31]:
dept_counts = df["Department"].value_counts()
dept_counts

Department
Computer Science    5
Economics           5
Physics             4
Mathematics         4
Psychology          3
Name: count, dtype: int64

**7. Which department has the largest difference between its highest and lowest marks?**

In [32]:
dept_spread = df.groupby("Department")["Marks"].agg(["max", "min"])
dept_spread["range"] = dept_spread["max"] - dept_spread["min"]
dept_spread.sort_values("range", ascending=False)

,max,min,range
Department,,,
Economics,87.0,65.0,22.0
Psychology,86.0,69.0,17.0
Mathematics,88.0,72.0,16.0
Physics,85.0,73.0,12.0
Computer Science,95.0,90.0,5.0


**8. Is the department with the highest average also the department containing the highest-scoring individual student?**

In [33]:
top_scorer = df.nlargest(1, "Marks")
top_scorer

,StudentID,Name,Age,City,Marks,Gender,Department,Result,Grade
6,7,Leela,20,Unknown,95.0,F,Computer Science,Pass,A


**9. How many students scored above their department's average?**

In [34]:
df["dept_avg"] = df.groupby("Department")["Marks"].transform("mean")
df["above_dept_avg"] = df["Marks"] > df["dept_avg"]
df["above_dept_avg"].sum()

np.int64(10)

In [35]:
df[["Name", "Department", "Marks", "dept_avg", "above_dept_avg"]]

,Name,Department,Marks,dept_avg,above_dept_avg
0,Asha,Physics,85.0,79.000000,True
1,Rahul,Computer Science,92.0,92.400000,False
2,Meera,Mathematics,82.0,79.000000,True
3,Vikram,Economics,68.0,73.600000,False
4,Simran,Mathematics,88.0,79.000000,True
5,Karan,Physics,73.0,79.000000,False
6,Leela,Computer Science,95.0,92.400000,True
7,Arjun,Economics,81.0,73.600000,True
8,Rekha,Psychology,69.0,79.333333,False
9,Dev,Mathematics,74.0,79.000000,False


**Gender distribution within each department (supporting detail for the summary):**

In [36]:
gender_by_dept = df.groupby(["Department", "Gender"]).size().unstack(fill_value=0)
gender_by_dept

Gender,F,M
Department,,
Computer Science,4,1
Economics,1,4
Mathematics,2,2
Physics,3,1
Psychology,2,1


**10. Three-sentence summary of the most important findings**

1. Computer Science is the strongest-performing department, with the highest
average mark (92.4), the tightest spread of scores (only 5 points between its
highest and lowest), and the individual top scorer, while Economics is the
weakest on average (73.6) and the most varied (a 22-point spread).
2. Female students averaged higher marks overall (85.4 vs. 75.0 for male
students), but this appears to be linked to department composition rather than
gender alone, since Computer Science is mostly female (4F/1M) and Economics is
mostly male (1F/4M).
3. Overall, 14 of 21 students (66.7%) passed, roughly half of all students
scored above their own department's average, and 18-year-olds had the highest
average performance (87.0) among age groups, though each age group contains
only a handful of students, so that particular pattern should be treated as
suggestive rather than conclusive.